In [1]:
from sklearn.datasets import fetch_california_housing, fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

def load_and_split(X, y, random_state=42):
    print(X.shape, y.shape)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def fetch_openml_numeric(name, target):
    ds = fetch_openml(name, as_frame=True)
    X = ds.data
    y = ds.target.astype(float)

    # оставляем только числовые признаки
    X = X.select_dtypes(include=[np.number])

    return X.values, y.values


def fetch_openml_numeric_by_id(data_id):
    ds = fetch_openml(data_id=data_id, as_frame=True)
    X = ds.data.select_dtypes(include=[np.number])
    y = ds.target.astype(float)
    return X.values, y.values

In [2]:
# "catboost-style (self-made, default catboost HP)",california_housing,0.842888,0.848189,849
# catboost (default HP),california_housing,0.842664,0.850693,872
# "catboost-style (self-made, default catboost HP)",bike_sharing,0.932619,0.924521,922
# catboost (default HP),bike_sharing,0.932571,0.924632,830
# "catboost-style (self-made, default catboost HP)",medical_charges,0.966301,0.975550,275
# catboost (default HP),medical_charges,0.966301,0.975550,275
# "catboost-style (self-made, default catboost HP)",king_county_house_prices,0.891442,0.858218,162
# catboost (default HP),king_county_house_prices,0.896683,0.871238,216


In [ ]:
RESULTS_PATH = "benchmark_results.csv"
N_ESTIMATORS = 1000
EARLY_STOPPING_ROUNDS = 20
OPTUNA_TRIALS = 100
LEARNING_RATE = 0.1

import os
import sys
sys.path.append("..")


import csv
from src.gradient_boosting_regressor import MyCatBoost

from catboost import CatBoostRegressor

def log_result_csv(model_name, name, r2_test, n_trees, path=RESULTS_PATH):
    file_exists = os.path.isfile(path)

    with open(path, mode="a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["model", "dataset", "r2_test", "n_trees"])
        writer.writerow([model_name, name, f"{r2_test:.6f}", n_trees])


# Experiment 1: Default Catboost
def test_base_catboost(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    gbrt = CatBoostRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        loss_function="RMSE",
        verbose=False
    )

    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    number_of_trees = gbrt.get_best_iteration() + 1
    
    log_result_csv(
        "Exp 1:  catboost (default HP)", name, r2, number_of_trees
    )

def test_my_catboost(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    def base_model_fn(noise_std):
        return CatBoostRegressor(
            iterations=1, 
            learning_rate=1.0,
            random_strength=noise_std,
            verbose=False
        )
    
    gbrt = MyCatBoost(
        base_model_fn=base_model_fn,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        loss_function="RMSE",
        random_strength=1,
        verbose=False
    )
    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    number_of_trees = len(gbrt.models)

    log_result_csv(
        "Exp 2:  catboost-style (self-made, default catboost HP)", 
        name, r2, number_of_trees
    )

import optuna

def test_my_catboost_random_hp(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value
    def base_model_fn(noise_std):
        # Создаём study с RandomSampler для каждого дерева
        study = optuna.create_study(
            direction="minimize",
            sampler=optuna.samplers.RandomSampler()
        )
        
        # Берём один trial (случайная выборка из search space)
        trial = study.ask()
        
        grow_policy = trial.suggest_categorical(
            "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
        )

        params = {
            "iterations": 1,
            "learning_rate": 1.0,
            "loss_function": "RMSE",
            "boost_from_average": False,
            "boosting_type": "Plain",
            "grow_policy": grow_policy,
            "border_count": trial.suggest_int("border_count", 32, 255),
            "feature_border_type": trial.suggest_categorical(
                "feature_border_type", ["GreedyLogSum", "Median", "Uniform"]
            ),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-6, 100.0, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 64),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 10.0, log=True
            ) * noise_std,
            "rsm": trial.suggest_float("rsm", 0.3, 1.0),
            "score_function": trial.suggest_categorical(
                "score_function", ["Cosine", "L2"]
            ),
            "random_seed": int.from_bytes(os.urandom(4), "little"),
            "verbose": False,
        }

        if grow_policy in ["SymmetricTree", "Depthwise"]:
            params["depth"] = trial.suggest_int("depth", 2, 12)
        else:
            params["max_leaves"] = trial.suggest_int("max_leaves", 8, 64)

        bootstrap_type = trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "No"]
        )
        params["bootstrap_type"] = bootstrap_type

        if bootstrap_type == "Bernoulli":
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
        elif bootstrap_type == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            )

        return CatBoostRegressor(**params)
    
    gbrt = MyCatBoost(
        base_model_fn=base_model_fn,
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        loss_function="RMSE",
        random_strength=1,
        verbose=False,
        fix_borders=False
    )
    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    number_of_trees = len(gbrt.models)

    log_result_csv(
        "Exp 3:  catboost-style (self-made, random HP)", 
        name, r2, number_of_trees
    )

# Exp 4: HPO Catboost
def test_base_catboost_HPO(name, value):
    X_train, X_val, X_test, y_train, y_val, y_test = value

    def objective(trial):
        params = {
            "n_estimators": N_ESTIMATORS,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "loss_function": "RMSE",
            "boosting_type": "Plain",
            "grow_policy": trial.suggest_categorical(
                "grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"]
            ),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "feature_border_type": trial.suggest_categorical(
                "feature_border_type", ["GreedyLogSum", "Median", "Uniform"]
            ),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-6, 100.0, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 64),
            "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 10.0, log=True
            ),
            "rsm": trial.suggest_float("rsm", 0.3, 1.0),
            "score_function": trial.suggest_categorical(
                "score_function", ["Cosine", "L2"]
            ),
            "random_seed": int.from_bytes(os.urandom(4), "little"),  # 32-bit entropy,
            "verbose": False,
        }

        if params["grow_policy"] in ["SymmetricTree", "Depthwise"]:
            params["depth"] = trial.suggest_int("depth", 2, 12)
        else:
            params["max_leaves"] = trial.suggest_int("max_leaves", 8, 64)

        bootstrap_type = trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli", "No"]
        )
        params["bootstrap_type"] = bootstrap_type

        if bootstrap_type == "Bernoulli":
            params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
        elif bootstrap_type == "Bayesian":
            params["bagging_temperature"] = trial.suggest_float(
                "bagging_temperature", 0.0, 10.0
            )

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        )

        pred = model.predict(X_val)
        r2 = r2_score(y_val, pred)
        return -r2  # minimize negative R2
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params

    gbrt = CatBoostRegressor(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        loss_function="RMSE",
        **best_params,
        verbose=False
    )

    gbrt.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )

    pred = gbrt.predict(X_test)
    r2 = r2_score(y_test, pred)
    number_of_trees = gbrt.get_best_iteration() + 1
    log_result_csv(
        "Exp 4:  catboost (HPO)", name, r2, number_of_trees
    )

def test_all_4(name, value):
    test_base_catboost(name, value)
    test_my_catboost(name, value)
    test_my_catboost_random_hp(name, value)
    test_base_catboost_HPO(name, value)

In [ ]:
import os
import numpy as np
from catboost import CatBoostRegressor

def base_model_fn(noise_std):
    seed = int.from_bytes(os.urandom(4), "little")  # 32-bit entropy
    return CatBoostRegressor(
        iterations=1,
        learning_rate=1.0,
        loss_function="RMSE",
        boost_from_average=False,
        boosting_type="Plain",
        random_seed=seed,
        random_strength=noise_std,
        verbose=False,
    )


model = MyCatBoost(
    base_model_fn=base_model_fn,
    n_estimators=1000,
    learning_rate=0.1,
    boosting_type="Plain",
    # bootstrap_type="MVS",
    # subsample=0.8,
    random_strength=1,
    bootstrap_type="No",
    # random_strength=0,
    verbose=10,
)

# Example usage on California Housing
cal = fetch_california_housing(as_frame=True)
X, y = cal.data.values, cal.target.values

#X, y = fetch_openml_numeric("Bike_Sharing_Demand", target="count")
X_train, X_val, X_test, y_train, y_val, y_test = load_and_split(
    X, y
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=20)
pred = model.predict(X_test)
r2_val = r2_score(y_val, model.predict(X_val))
print(f"Val R2: {r2_val:.6f}")
r2 = r2_score(y_test, pred)
print(f"Test R2: {r2:.6f}")

(20640, 8) (20640,)
/tmp/tmpr48__mtn.borders
)____
[0] train RMSE=1.084800, val RMSE=1.109309
[10] train RMSE=0.720305, val RMSE=0.741407
[20] train RMSE=0.602846, val RMSE=0.623887
[30] train RMSE=0.551893, val RMSE=0.574570
[40] train RMSE=0.522827, val RMSE=0.549172
[50] train RMSE=0.503542, val RMSE=0.535744
[60] train RMSE=0.490299, val RMSE=0.526814
[70] train RMSE=0.478378, val RMSE=0.519120
[80] train RMSE=0.466512, val RMSE=0.511988
[90] train RMSE=0.457544, val RMSE=0.506832
[100] train RMSE=0.450010, val RMSE=0.503081
[110] train RMSE=0.443456, val RMSE=0.500510
[120] train RMSE=0.436755, val RMSE=0.497391
[130] train RMSE=0.430456, val RMSE=0.493865
[140] train RMSE=0.424628, val RMSE=0.492648
[150] train RMSE=0.419323, val RMSE=0.490293
[160] train RMSE=0.414076, val RMSE=0.488318
[170] train RMSE=0.409155, val RMSE=0.486342
[180] train RMSE=0.404739, val RMSE=0.483865
[190] train RMSE=0.399681, val RMSE=0.482842
[200] train RMSE=0.395822, val RMSE=0.481777
[210] train RMS

In [28]:
model = CatBoostRegressor(
    n_estimators=2000,
    learning_rate=0.1,
    loss_function="RMSE",
    boosting_type="Plain",
    bootstrap_type="MVS",
    subsample=0.8,
    depth=7,
    random_strength=1,
    # bootstrap_type="No",
    # random_strength=0,
    random_seed=7,
    verbose=False
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=20,
)
pred = model.predict(X_test)
r2 = r2_score(y_test, pred)
r2_val = r2_score(y_val, model.predict(X_val))
print(f'No trees: {model.tree_count_}')
print(f"CatBoost Val R2: {r2_val:.6f}")
print(f"CatBoost Test R2: {r2:.6f}")

No trees: 988
CatBoost Val R2: 0.847251
CatBoost Test R2: 0.849956
